## Exercise: Final EDA Workflow

You are a data scientist working for a financial services company. The company wants to predict which customers are likely to miss payment within the next 60 days. You receive a dataset containing:
- Customer demographics.
- Accounot balances.
- Historical payment records.
- Transaction activity.
- Credit utilization.
- Customer service interactions.
- Account status.
- Payment delinquency outcomes.

Your task is to design a complete exploratory data analysis and preprocessing workflow.

### Project Requirements

Chapter 5 Capstone Checklist
- Problem definition: Define the target, observation unit, prediction timestamp, and success metrics.
- Dataset inspection: Identify features, data types, identifiers, and structural issues.
- Data quality audit: Check missingness, duplicates, invalid values, and inconsistent categories.
- Univariate EDA: Analyze numerical and categorical distributions.
- Multivariate EDA: Investigate correlations, feature relationships, and interactions.
- Target analysis: Evaluate class imbalance and relationships with candidate predictors.
- Leakage audit: Identify unavailable future information and invalid feature construction.
- Evaluation design: Choose a train-validation-test strategy appropriate for 60-day outcomes.
- Preprocessing pipeline: Implement leakage-safe imputation, encoding, and scaling.
- Baseline modeling: Train a baseline model and evaluate appropriate metrics.
- Documentation: Summarize findings, assumptions, limitations, and recommended next steps.

### Expected Deliverables

A complete submission should include a reproducible Jupyter notebook, a documented preprocessing pipeline, appropriate evaluation results, and a written summary of findings.

A strong submission should explain not only what was done, but why each decision was appropriate for the business problem and data structure.

### Stage 1 — Problem Definition

Define the target, observation unit, prediction timestamp, and success metrics. The example dataset has been loaded below.

**Business objective:** Predict which customers are likely to miss at least one scheduled payment within the next 60 days, enabling the financial services company to prioritize preventive outreach.

**Observation unit:** One customer at a monthly snapshot date.

**Prediction timestamp:** The first day of each month, using only information available at that time.

**Target:** A binary variable indicating whether the customer misses at least one scheduled payment during the subsequent 60 days.

**Evaluation:** Use average precision and ROC-AUC to evaluate ranking, precision and recall at operational thresholds, and potentially precision@K for a capacity-constrained outreach program.

**Validation strategy:** Chronological train-validation-test splitting, with safeguards ensuring labels are fully observable at each historical training cutoff.

### Stage 2 — Building and Inspecting Our Dataset

Identify features, data types, identifiers, and structural issues.

#### 2.1 — Dataset Design
We'll simulate 1,000 customers observed monthly for 24 months, producing 24,000 customer-month records. Each row will represent the information available about a customer at a monthly prediction snapshot. Our target will be whether the customer misses at least one scheduled payment in the following 60 days.

| Feature | Description | Type |
| :--- | :--- | :--- |
| `customer_id` | Unique customer identifier | Identifier |
| `snapshot_date` | Monthly prediction timestamp | Datetime |
| `age` | Customer age | Numerical |
| `annual_income` | Estimated annual income | Numerical |
| `account_balance` | Outstanding account balance | Numerical |
| `credit_utilization` | Proportion of available credit used | Numerical |
| `transactions_30d` | Number of transactions in the preceding 30 days | Numerical, discrete |
| `late_payments_12m` | Number of historical late payments | Numerical, discrete |
| `account_tenure_months` | Months since account opening | Numerical, discrete |
| `employment_status` | Employment category | Categorical |
| `region` | Customer region | Categorical |
| `missed_payment_date` | Future missed-payment outcome | Binary target |

We'll also introduce a small number of deliberate data-quality problems in a later stage so you can practice identifying them rather than being told exactly where they are.

**Important modeling distinction**

This dataset will simulate a coherent 60-day outcome process from underlying monthly payment events. That matters because consecutive monthly snapshots have overlapping prediction horizons. For example, a missed payment in March could legitimately produce a positive label for both a February and March snapshot. We should not independently generate unrelated 60-day labels for every monthly row.

#### 2.2 — Creating the Dataset
The generator creates historical missed-payment events and derives each 60-day target from those events. It also uses a fixed random seed so the dataset is reproducible.

Two limitations to remember: this is synthetic data, and the historical features are simulated summaries rather than features reconstructed from a complete transaction ledger. We'll use it to learn the workflow, not to draw real financial conclusions.

#### 2.3 — Set Up Your Notebook
Place the CSV in the same folder as your notebook and load the data into the notebook.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PATH = './../datasets/financial_customer_months.csv'

def readData(path):    
    dataframe = pd.read_csv(
        path
    )
    return dataframe

df = readData(PATH)

#### 2.4 — Structural Inspection

In Chapter 5.2, we discussed understanding the dataset before making transformations. I want you to write Python code to answer these questions.

**Task A — Basic structure**
1. How many rows and columns are in the dataset?
2. What are the first five observations?
3. What are the data types of each column?
4. How many non-null observations does each column contain?
5. What descriptive statistics can we obtain for numerical and categorical features?

Hints: Think about the Pandas methods shape, head(), dtypes, info(), and describe().

**Task B — Feature classification**

Identify which columns are:
- Numerical continuous.
- Numerical discrete.
- Categorical nominal.
- Identifiers.
- Datetime.
- Target.

A feature being stored as an integer does not necessarily mean it should be treated as a numerical predictor. For example, customer_id is stored numerically, but its magnitude has no meaningful quantitative relationship with payment risk.

**Task C — Understand the observation unit**

Write code to answer:
1. How many unique customers exist?
2. How many unique snapshot dates exist?
3. How many observations exist per customer?
4. What is the earliest snapshot date?
5. What is the latest snapshot date?

Then answer this conceptual question: Why would counting duplicated customer_id values be misleading when determining whether this dataset contains duplicate observations?

**Task D — Identify a potential data type problem**

Inspect the data type of snapshot_date. Pandas often reads dates from CSV files as strings. What would you do to ensure this column is represented as a proper datetime type, and why does that matter for our later temporal splitting?

In [10]:
# Task A - Basic Structure
print('---Basic Structure of the Dataset---')

print('\n---Shape of the dataset:---')
print(df.shape)

print('\n---First 5 rows of data:---')
print(df.head())

print('\n---Datatype of each column:---')
print(df.dtypes)

print('\n---Number of non-null observations in each column:---')
df.info()

print('\n---Descriptive statistics for numerical and categorical features:---')
df.describe(include='all')

---Basic Structure of the Dataset---

---Shape of the dataset:---
(24000, 12)

---First 5 rows of data:---
   customer_id snapshot_date  age  annual_income  account_balance  \
0        10001    2024-01-01   47       42481.25          2231.11   
1        10002    2024-01-01   61       69380.81          5855.66   
2        10003    2024-01-01   34       52645.80          6305.26   
3        10004    2024-01-01   28       44475.26          5096.54   
4        10005    2024-01-01   71       45374.93          2588.58   

   credit_utilization  transactions_30d  late_payments_12m  \
0              0.2342                19                  0   
1              0.4547                14                  0   
2              0.5199                20                  0   
3              0.2667                26                  0   
4              0.2399                19                  0   

   account_tenure_months employment_status     region  missed_payment_60d  
0                     42     

,customer_id,snapshot_date,age,annual_income,account_balance,credit_utilization,transactions_30d,late_payments_12m,account_tenure_months,employment_status,region,missed_payment_60d
count,24000.000000,24000,24000.000000,24000.00000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000,24000,24000.000000
unique,NaN,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,4,NaN
top,NaN,2024-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Employed,Northeast,NaN
freq,NaN,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16800,6144,NaN
mean,10500.500000,NaN,48.313000,77204.48691,4688.554498,0.397754,18.389042,0.302458,72.598000,NaN,NaN,0.067292
std,288.681005,NaN,16.164341,40283.67506,2421.885488,0.207989,4.842606,0.586934,35.877133,NaN,NaN,0.250532
min,10001.000000,NaN,21.000000,18000.00000,0.000000,0.000000,4.000000,0.000000,1.000000,NaN,NaN,0.000000
25%,10250.750000,NaN,34.000000,48926.50250,2917.317500,0.241400,15.000000,0.000000,42.000000,NaN,NaN,0.000000
50%,10500.500000,NaN,48.500000,68299.47000,4599.345000,0.385600,18.000000,0.000000,73.000000,NaN,NaN,0.000000
75%,10750.250000,NaN,63.000000,95618.50250,6345.735000,0.543100,21.000000,0.000000,103.000000,NaN,NaN,0.000000


The target variable has a mean of approximately:
$$
\bar y = 0.067292
$$
Because `missed_payment_60d` is binary, its mean equals the proportion of positive observations.

Therefore:
$$
P(Y = 1) \approx 6.73\%
$$
This tells us that missed payments are relatively rare in our datset. We'll investigate that more thoroughly during target-distribution analysis.


In [11]:
# Task B - Feature Classification
identifier_cols = ['customer_id']

datetime_cols = ['snapshot_date']

continuous_cols = [
    'annual_income',
    'account_balance',
    'credit_utilization'
]

discrete_cols = [
    'age',
    'transactions_30d',
    'late_payments_12m',
    'account_tenure_months'
]

categorical_cols = [
    'employment_status',
    'region'
]

target_col = 'missed_payment_60d'

In [ ]:
# Task C - Understand the Oberservation Unit
print('---Understanding the Observational Unit---')

print('\n---Number of unique customers:---')
print(df['customer_id'].nunique())

print('\n---Number of unique snapshot dates:---')
print(df['snapshot_date'].nunique())

print('\n---Number of observations per customer:---')
print(df['customer_id'].value_counts())

print('---\nEarliest snapshot date:---')
print(df['snapshot_date'].min())

print('---\nLatest snapshot date:---')
print(df['snapshot_date'].max())

---Understanding the Observational Unit---

---Number of Unique Customers:---
1000

---Number of Unique Snapshot Dates:---
24

---Number of Observations Per Customer:---
customer_id
10001    24
10002    24
10003    24
10004    24
10005    24
         ..
10996    24
10997    24
10998    24
10999    24
11000    24
Name: count, Length: 1000, dtype: int64
---
Earliest Snapshot Date:---
2024-01-01
---
Latest Snapshot Date:---
2025-12-01


In [14]:
# Task D - Identify a Potential Data Type Problem
print('---Identifying Potential Data Type Problems---')

print('\n---Inspecting the data type of snapshot_date:---')
df['snapshot_date'] = pd.to_datetime(
    df['snapshot_date']
)

print(df['snapshot_date'].dtype)

print('\n---Inspecting duplicate of customer-month records:---')
duplicate_count = df.duplicated(
    subset=['customer_id', 'snapshot_date']
).sum()

print(duplicate_count)

---Identifying Potential Data Type Problems---

---Inspecting the data type of snapshot_date:---
datetime64[us]

---Inspecting duplicate of customer-month records:---
0


#### Interpretation of the Dataset Structure

1. There's 24,000 observations in the dataset representing the 24 customer-month records for each of the 1000 customers.
2. `customer_id` is not an appropriate numerical predictor because it's an identifier. The feature simply identifies individual customers to determine who the record belongs too. Additionally, `customer_id` numerical magnitude is arbitrary which is why we generally exclude it as a numerical predictor.
3. The combination of `customer_id` and `snapshot_date` is more important for duplicate detection than either column individually because individually those columns will have duplicate values that showup naturally from the dataset description. Multiple observations will have the same `customer_id` because the dataset contains information on that customer in multiple months. Multiple oberservations will have the same `snapshot_date` because the dataset contains information on customers' financials from the same month. The combination of the two columns having duplicate values could pose a problem since we shouldn't expect the same customer to have financial information from the same date.
4. We must convert `snapshot_date` before implementing our chronological validation strategy so that we can perform meaningful time-based operations. Relying on strings is fragile and prevents us from using Pandas' full datetime functionality.

### Stage 3 — Data Quality Audit

Check missingness, duplicates, invalid values, and inconsistent categories.

#### 3.1 — What We'll Investigate

Before we visualize relationships or build predictive models, we need to determine whether the dataset is trustworthy. A model can execute successfully and produce convincing metrics while learning from invalid or inconsistent data. Our objective is to identify these issues before making cleaning decisions

| Dimension | Question |
| :--- | :--- |
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |

### Stage 4 — Univariate EDA 

Analyze numerical and categorical distributions.

### Stage 5 — Multivariate EDA 

Investigate correlations, feature relationships, and interactions.

### Stage 6 — Target analysis 

Evaluate class imbalance and relationships with candidate predictors.

### Stage 7 — Leakage Audit

Identify unavailable future information and invalid feature construction. 

### Stage 8 — Evaluation Design

Choose a train-validation-test strategy appropriate for 60-day outcomes.